# Module 05 - Improving RAG

**Duration:** 60 minutes

A basic RAG pipeline works, but it has predictable failure modes.
This module covers four techniques that address the most common ones.
Each technique has a before/after comparison so you can see the difference directly.

---


## Setup


In [ ]:
from ragsst.ragtool import RAGTool
import requests, json
from os import getenv
from urllib.parse import urljoin

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')
MODEL = 'llama3.2'

tool = RAGTool(data_path='../data/sample_docs', collection_name='workshop_docs')
tool.setup_vec_store()
print('Ready.')


In [ ]:
def generate(prompt: str, temp: float = 0.3) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': MODEL, 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


def basic_rag(query: str, n: int = 3) -> str:
    context = tool.get_relevant_text(query, nresults=n)
    prompt = tool.get_context_prompt(query, context)
    return generate(prompt)


---

## Technique 1: Re-ranking

Vector search retrieves the top-k most similar chunks by embedding distance.
But embedding similarity is not the same as relevance to a specific question.
A chunk can be topically related without actually answering what was asked.

A cross-encoder solves this. It takes a query-document pair and outputs
a relevance score by reading both together, which is much more accurate
than comparing independent embeddings. The downside is that it is slower,
so we only use it to re-rank the top-k candidates already retrieved.


In [ ]:
from sentence_transformers import CrossEncoder

# This model is specifically trained for passage re-ranking
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


def rag_with_reranking(query: str, retrieve_n: int = 10, rerank_to: int = 3) -> str:
    # Step 1: retrieve more candidates than we need
    results = tool.collection.query(query_texts=[query], n_results=retrieve_n)
    candidates = results['documents'][0]

    # Step 2: score each candidate against the query
    pairs = [[query, doc] for doc in candidates]
    scores = reranker.predict(pairs)

    # Step 3: sort by score and keep the top ones
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in ranked[:rerank_to]]

    context = '\n'.join(top_docs)
    return generate(tool.get_context_prompt(query, context))


query = 'What workshops does the AI service center run on Tuesdays?'

print('Without reranking:')
print(basic_rag(query, n=3))

print('\nWith reranking (retrieve 10, rerank to 3):')
print(rag_with_reranking(query))


The cross-encoder reads the query and each chunk together, so it can catch
cases where the embedding similarity was misleading.

**Exercise:** Find a question where reranking makes no difference.
Then find one where it clearly helps. What characterises the two cases?


---

## Technique 2: HyDE (Hypothetical Document Embeddings)

A question and its answer are often expressed very differently.
'When does the paper reading session take place?' does not look much like
'The paper reading sessions take place every Wednesday afternoon.'

HyDE flips the retrieval query: instead of embedding the question,
we ask the LLM to write a hypothetical answer, then embed that instead.
An answer looks like an answer, so it is more likely to land close
to the actual answer in the vector space.


In [ ]:
def generate_hypothetical_answer(query: str) -> str:
    prompt = (
        'Write a short, factual passage that would directly answer the following question. '
        'Do not say you are guessing. Just write the passage as if it were from a document.\n\n'
        f'Question: {query}\n'
        'Passage:'
    )
    return generate(prompt, temp=0.5)


def rag_with_hyde(query: str, n: int = 3) -> str:
    # Generate a hypothetical answer
    hypothetical = generate_hypothetical_answer(query)

    # Retrieve using the hypothetical answer instead of the question
    context = tool.get_relevant_text(hypothetical, nresults=n)

    # Answer using the original question and the retrieved context
    return generate(tool.get_context_prompt(query, context))


query = 'When do the paper reading sessions take place?'

print('Hypothetical answer generated by LLM:')
print(generate_hypothetical_answer(query))

print('\nWithout HyDE:')
print(basic_rag(query))

print('\nWith HyDE:')
print(rag_with_hyde(query))


HyDE adds an extra LLM call, which increases latency.
It is most useful when questions are short or under-specified.

**Exercise:** Try HyDE on a question where the basic RAG already works well.
Does HyDE make things worse? Can you find a case where it retrieves the wrong document?


---

## Technique 3: Multi-Query

A single query has a single embedding, which searches a single direction in vector space.
If the relevant document uses slightly different vocabulary, it might not come up.

Multi-query generates several rephrasings of the question and retrieves for each one.
The results are merged (with duplicates removed) before generating the answer.
This increases recall at the cost of more retrieval calls.


In [ ]:
def generate_query_variants(query: str, n: int = 3) -> list[str]:
    prompt = (
        f'Write {n} different versions of the following question. '
        'Each version should ask for the same information but use different words. '
        'Output only the questions, one per line, with no numbering or extra text.\n\n'
        f'Question: {query}'
    )
    response = generate(prompt, temp=0.7)
    variants = [line.strip() for line in response.strip().split('\n') if line.strip()]
    return variants[:n]


def rag_with_multiquery(query: str, n_variants: int = 3, n_results_each: int = 2) -> str:
    variants = generate_query_variants(query, n_variants)
    all_queries = [query] + variants

    # Retrieve for each query variant and deduplicate
    seen = set()
    all_docs = []
    for q in all_queries:
        results = tool.collection.query(query_texts=[q], n_results=n_results_each)
        for doc in results['documents'][0]:
            if doc not in seen:
                seen.add(doc)
                all_docs.append(doc)

    context = '\n\n'.join(all_docs)
    return generate(tool.get_context_prompt(query, context))


query = 'What free resources does the AI center provide?'

print('Query variants:')
for v in generate_query_variants(query):
    print(' -', v)

print('\nWithout multi-query:')
print(basic_rag(query))

print('\nWith multi-query:')
print(rag_with_multiquery(query))


---

## Technique 4: RAG-Fusion

RAG-Fusion is a combination of multi-query and smarter result merging.
Instead of just deduplicating the results from multiple queries,
it uses Reciprocal Rank Fusion to score each document based on how
highly it ranked across all the queries.

A document that came top-3 in all three query variants gets a much higher
combined score than one that appeared once in a lower position.


In [ ]:
def reciprocal_rank_fusion(ranked_lists: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return [doc for doc, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]


def rag_fusion(query: str, n_variants: int = 3, n_per_query: int = 5, final_n: int = 3) -> str:
    variants = generate_query_variants(query, n_variants)
    all_queries = [query] + variants

    # Get a ranked list of docs for each query
    ranked_lists = []
    for q in all_queries:
        results = tool.collection.query(query_texts=[q], n_results=n_per_query)
        ranked_lists.append(results['documents'][0])

    # Fuse the ranked lists
    fused = reciprocal_rank_fusion(ranked_lists)

    context = '\n\n'.join(fused[:final_n])
    return generate(tool.get_context_prompt(query, context))


query = 'What free resources does the AI center provide?'

print('Without RAG-Fusion:')
print(basic_rag(query))

print('\nWith RAG-Fusion:')
print(rag_fusion(query))


---

## Summary

| Technique | What it fixes | Cost |
| --- | --- | --- |
| Re-ranking | Retrieved chunks that are topically close but not actually relevant | Extra model call per candidate |
| HyDE | Questions that look very different from their answers | One extra LLM call |
| Multi-query | Single query missing relevant docs due to vocabulary mismatch | N extra retrieval calls |
| RAG-Fusion | Same as multi-query, with smarter result merging | N extra retrieval calls |

None of these is always better. The right choice depends on your documents and query patterns.
That is what Module 06 is about.

---

**Exercises**

1. Combine re-ranking with multi-query: retrieve using multiple queries,
   then pass all candidates to the cross-encoder and keep the top 3.

2. Try HyDE on a query about Wild Tales. Does the hypothetical answer
   mention anything not in the document? What happens to retrieval if it does?

3. For RAG-Fusion, print the fused ranking and compare it to the ranking
   from any single query. Are they different? Which ones moved up?

---

**Further reading**

- HyDE paper: https://arxiv.org/abs/2212.10496
- RAG-Fusion post: https://towardsdatascience.com/forget-rag-the-future-is-rag-fusion-1147298d8ad1
- Cross-encoders vs bi-encoders: https://www.sbert.net/docs/cross_encoder/usage.html
